# 🔬 Shot Peening Surface Deformation Detection
### Training on Google Colab (Free T4 GPU)

**Steps:**
1. Run all cells top to bottom
2. Upload your annotated dataset as a ZIP
3. Train the model (~1-2 hours on T4 GPU)
4. Download the trained model (`best_model.pth`)
5. Use it locally with `PREDICT.bat`

> ⚡ **Runtime → Change runtime type → T4 GPU** before running!

## Step 1: Check GPU

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU')

## Step 2: Install Dependencies

In [ ]:
%%capture
!pip install segmentation-models-pytorch albumentations tifffile

## Step 3: Clone Your GitHub Repo

In [ ]:
!git clone -b ai-model https://github.com/Sopan777/Shot-Peening.git shot-peening-ai
%cd shot-peening-ai
!ls

## Step 4: Upload Your Dataset

Upload a ZIP file containing your annotated data with this structure:
```
dataset.zip
├── train/
│   ├── images/   (.tif files)
│   └── masks/    (.png binary masks)
├── val/
│   ├── images/
│   └── masks/
└── test/
    ├── images/
    └── masks/
```
> Run `PREPARE_DATA.bat` locally first to create this structure, then ZIP the `data/` folder.

In [ ]:
from google.colab import files
import zipfile, os

print('Select your dataset ZIP file...')
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as zf:
    zf.extractall('data/')

print('\n✅ Dataset extracted. Structure:')
for root, dirs, fs in os.walk('data'):
    level = root.replace('data', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        sub = ' ' * 2 * (level + 1)
        print(f'{sub}{len(fs)} files')

## Step 5: Train the Model

In [ ]:
# ── Configuration ──────────────────────────────────────────
ENCODER     = 'resnet34'   # backbone
IMG_SIZE    = 512          # input resolution
BATCH_SIZE  = 16           # T4 can handle 16 at 512x512
EPOCHS      = 100
PATIENCE    = 15           # early stopping
DATA_DIR    = 'data'
OUTPUT_DIR  = 'outputs'
# ──────────────────────────────────────────────────────────

In [ ]:
import sys
sys.path.insert(0, 'src')

import os, random, time
import numpy as np
import torch
import torch.nn as nn
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from tqdm.notebook import tqdm

from dataset import get_dataloaders
from model import create_model, set_encoder_trainable, get_parameter_groups, get_model_summary
from utils import calculate_metrics, MetricTracker, save_checkpoint, setup_logging

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Loss functions
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits).view(-1)
        t = targets.view(-1)
        inter = (probs * t).sum()
        return 1.0 - (2.*inter + self.smooth) / (probs.sum() + t.sum() + self.smooth)

class CombinedLoss(nn.Module):
    def __init__(self, bce_w=0.5, dice_w=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.bce_w, self.dice_w = bce_w, dice_w
    def forward(self, logits, targets):
        return self.bce_w * self.bce(logits, targets) + self.dice_w * self.dice(logits, targets)

In [ ]:
# Build model + dataloaders
dataloaders = get_dataloaders(DATA_DIR, batch_size=BATCH_SIZE, img_size=IMG_SIZE, num_workers=2)
train_loader = dataloaders['train']
val_loader   = dataloaders['val']
test_loader  = dataloaders.get('test')

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

model = create_model(architecture='Unet', encoder_name=ENCODER, encoder_weights='imagenet', classes=1).to(device)
get_model_summary(model)

In [ ]:
import os
os.makedirs(f'{OUTPUT_DIR}/models', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/logs', exist_ok=True)

BEST_MODEL_PATH = f'{OUTPUT_DIR}/models/best_model.pth'

# Stage 1: freeze encoder
set_encoder_trainable(model, trainable=False)
param_groups = get_parameter_groups(model, encoder_lr=1e-4, decoder_lr=5e-4)
optimizer  = AdamW(param_groups, weight_decay=1e-4)
scheduler  = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
scaler     = GradScaler()
criterion  = CombinedLoss()

history = {'train_loss': [], 'val_loss': [], 'val_iou': [], 'val_dice': []}
best_iou, patience_counter, freeze_epochs = 0.0, 0, 30

start = time.time()
for epoch in range(1, EPOCHS + 1):
    # Stage transition
    if epoch == freeze_epochs + 1:
        set_encoder_trainable(model, trainable=True)
        optimizer.param_groups[0]['lr'] = 1e-4
        optimizer.param_groups[1]['lr'] = 5e-4
        print('\n⚡ Stage 2: Encoder unfrozen — fine-tuning all layers')
    if epoch <= freeze_epochs:
        optimizer.param_groups[1]['lr'] = 1e-3

    # ── Train ──
    model.train()
    t_tracker = MetricTracker()
    for imgs, masks in tqdm(train_loader, desc=f'Ep {epoch:3d} Train', leave=False):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad(set_to_none=True)
        with autocast():
            logits = model(imgs)
            loss   = criterion(logits, masks)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        m = calculate_metrics(logits.detach(), masks.detach())
        m['loss'] = loss.item()
        t_tracker.update(m, n=imgs.size(0))

    # ── Validate ──
    model.eval()
    v_tracker = MetricTracker()
    with torch.no_grad():
        for imgs, masks in tqdm(val_loader, desc=f'Ep {epoch:3d} Val  ', leave=False):
            imgs, masks = imgs.to(device), masks.to(device)
            with autocast():
                logits = model(imgs)
                loss   = criterion(logits, masks)
            m = calculate_metrics(logits, masks)
            m['loss'] = loss.item()
            v_tracker.update(m, n=imgs.size(0))

    scheduler.step()
    t = t_tracker.get_averages()
    v = v_tracker.get_averages()
    history['train_loss'].append(t['loss'])
    history['val_loss'].append(v['loss'])
    history['val_iou'].append(v['iou'])
    history['val_dice'].append(v['dice'])

    print(f"Ep {epoch:3d} | train_loss={t['loss']:.4f} | val_loss={v['loss']:.4f} | IoU={v['iou']:.4f} | Dice={v['dice']:.4f}")

    if v['iou'] > best_iou:
        best_iou = v['iou']
        patience_counter = 0
        save_checkpoint(model, optimizer, epoch, v, BEST_MODEL_PATH)
        print(f'   ★ Best model saved (IoU={best_iou:.4f})')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

elapsed = (time.time() - start) / 60
print(f'\n✅ Training done in {elapsed:.1f} min | Best Val IoU: {best_iou:.4f}')

## Step 6: Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'],   label='Val Loss')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')

axes[1].plot(history['val_iou'],  label='Val IoU')
axes[1].plot(history['val_dice'], label='Val Dice')
axes[1].set_title('Metrics'); axes[1].legend(); axes[1].set_xlabel('Epoch')

plt.tight_layout()
plt.savefig('outputs/training_curves.png', dpi=150)
plt.show()

## Step 7: Test Set Evaluation

In [ ]:
if test_loader:
    ckpt = torch.load(BEST_MODEL_PATH, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    test_tracker = MetricTracker()
    with torch.no_grad():
        for imgs, masks in tqdm(test_loader, desc='Testing'):
            imgs, masks = imgs.to(device), masks.to(device)
            with autocast():
                logits = model(imgs)
                loss   = criterion(logits, masks)
            m = calculate_metrics(logits, masks)
            m['loss'] = loss.item()
            test_tracker.update(m, n=imgs.size(0))
    r = test_tracker.get_averages()
    print('\n=== TEST RESULTS ===')
    for k, v in r.items():
        print(f'  {k:20s}: {v:.4f}')

## Step 8: Download Trained Model
Download `best_model.pth` to your local machine, then place it in `outputs/models/` and run `PREDICT.bat`.

In [ ]:
from google.colab import files
print('Downloading best_model.pth...')
files.download(BEST_MODEL_PATH)
print('\n✅ Done! Place the file in:  outputs/models/best_model.pth')
print('Then run PREDICT.bat on your local machine.')